# Импорты и глобальные параметры

In [40]:
import os
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import pandas as pd
import string
import random

import inspect
if not hasattr(inspect, 'getargspec'):
    def getargspec(func):
        spec = inspect.getfullargspec(func)
        return spec.args, spec.varargs, spec.varkw, spec.defaults
    inspect.getargspec = getargspec


import pymorphy2  # Импорт библиотеки для лемматизации

# Инициализация морфологического анализатора
morph = pymorphy2.MorphAnalyzer()

CHUNK_SIZE = 500
STEP_SIZE = 200
EPOCHS = 70
LR = 0.00136
BATCH_SIZE = 64
VAL_SIZE = 0.2
# Путь к файлу с предобученными эмбеддингами
EMBEDDING_FILE = "model.txt"

# Токенизация и лемматизация

In [41]:
def tokenize(text):
    """
    Извлекает из текста последовательности символов, состоящие из латинских или кириллических букв и цифр,
    приводит их к нижнему регистру и лемматизирует.
    Это помогает отбросить знаки препинания и спецсимволы, а лемматизация позволяет увеличить покрытие эмбеддингов.
    """
    # Извлекаем токены (слова) из текста
    tokens = re.findall(r'[a-zа-яё0-9]+', text.lower())
    # Лемматизация токенов с использованием pymorphy2
    lemmas = [morph.parse(token)[0].normal_form for token in tokens]
    return lemmas

# Функции анализа покрытия эмбеддингов

In [42]:
def check_embedding_coverage(texts, embeddings):
    """
    Функция оценивает покрытие эмбеддингов в списке текстов.
    Она вычисляет:
      - Долю всех токенов (с учётом повторений), которые найдены в словаре эмбеддингов.
      - Долю уникальных токенов, найденных в словаре.
    Также выводит топ-10 токенов, отсутствующих в эмбеддингах, с их частотами.
    """
    total_tokens = 0
    covered_tokens = 0
    missing_tokens_counter = Counter()
    all_tokens = []  # для уникальных токенов

    for text in texts:
        tokens = tokenize(text)
        all_tokens.extend(tokens)
        total_tokens += len(tokens)
        for token in tokens:
            if token in embeddings:
                covered_tokens += 1
            else:
                missing_tokens_counter[token] += 1

    overall_coverage = covered_tokens / total_tokens if total_tokens > 0 else 0

    unique_tokens = set(all_tokens)
    unique_total = len(unique_tokens)
    unique_covered = sum(1 for token in unique_tokens if token in embeddings)
    unique_coverage = unique_covered / unique_total if unique_total > 0 else 0

    print("=== Отчет по покрытию эмбеддингов ===")
    print(f"Общее количество токенов: {total_tokens}")
    print(f"Количество найденных токенов: {covered_tokens}")
    print(f"Покрытие эмбеддингов (по частотам): {overall_coverage:.2%}")
    print(f"Всего уникальных токенов: {unique_total}")
    print(f"Найдено уникальных токенов: {unique_covered}")
    print(f"Покрытие эмбеддингов (уникальные токены): {unique_coverage:.2%}")
    
    # Выводим топ-10 отсутствующих токенов
    if missing_tokens_counter:
        print("\nТоп-10 отсутствующих токенов:")
        for token, count in missing_tokens_counter.most_common(10):
            print(f"{token}: {count}")
    else:
        print("\nВсе токены найдены в словаре эмбеддингов.")

# Загрузка предобученных эмбеддингов

In [43]:
def load_pretrained_embeddings(filepath):
    embeddings = {}
    with open(filepath, "r", encoding="utf-8") as f:
        header = f.readline().strip()  # например, "249946 300"
        _, emb_dim = header.split()
        emb_dim = int(emb_dim)
        for line in f:
            parts = line.rstrip().split()
            token = parts[0]
            # Убираем постфикс (например, _PUNCT, _ADP, _CCONJ и т.д.) и приводим к нижнему регистру
            token = re.sub(r'_[A-Z]+$', '', token).lower()
            vector = np.array([float(x) for x in parts[1:]], dtype=np.float32)
            embeddings[token] = vector
    return embeddings, emb_dim

# Загружаем эмбеддинги и приводим ключи к нижнему регистру
embeddings_dict, EMBEDDING_DIM = load_pretrained_embeddings(EMBEDDING_FILE)
print(f"Загружено {len(embeddings_dict)} эмбеддингов размерности {EMBEDDING_DIM}")

Загружено 192367 эмбеддингов размерности 300


# Вспомогательные функции: чтение, нарезка, формирование усредненного по чанку вектора

In [44]:
# Чтение файла
def read_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return f.read()

# Разбиение текста на отрывки
def chunk_text(text, chunk_size=200, step=200):
    tokens = text.split()
    chunks = []
    start = 0
    while start < len(tokens):
        chunk = tokens[start:start+chunk_size]
        if not chunk:
            break
        chunks.append(" ".join(chunk))
        start += step
    return chunks

# Функция для подготовки фиксированных чанков текста от разных авторов
def prepare_author_chunks(author_files, chunk_size=200, step=200):
    texts, labels, label2author = [], [], {}
    # Перебираем все файлы, соответствующие авторам
    for label_idx, filepath in enumerate(author_files):
        # Извлекаем имя автора из имени файла
        author_name = os.path.splitext(os.path.basename(filepath))[0]
        label2author[label_idx] = author_name
        # Разбиваем текст автора на чанки фиксированного размера и шага
        chunks = chunk_text(read_file(filepath), chunk_size, step)
        # Добавляем полученные чанки в общий список текстов
        texts.extend(chunks)
        # Добавляем соответствующую метку (label_idx) для каждого чанка
        labels.extend([label_idx] * len(chunks))

    # Возвращаем список текстов, список меток и отображение меток в имена авторов
    return texts, labels, label2author

def compute_average_vector(text, embeddings, embedding_dim):
    # Используем токенизацию с лемматизацией
    tokens = tokenize(text)
    vectors = []
    for token in tokens:
        if token in embeddings:
            vectors.append(embeddings[token])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        # Если ни один токен не найден, вернуть нулевой вектор
        return np.zeros(embedding_dim)

# Классы Dataset и MLP

In [45]:
class TextDataset(Dataset):
    def __init__(self, X_vectors, y_labels):
        # X_vectors – numpy массив размерности (n_samples, embedding_dim)
        self.X = torch.tensor(X_vectors, dtype=torch.float32)
        self.y = torch.tensor(y_labels, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# MLP с двумя скрытыми слоями
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim1=100, hidden_dim2=50, output_dim=6, dropout_rate=0.379):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.fc3 = nn.Linear(hidden_dim2, output_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# Функции обучения и предсказания

In [46]:
def train_model(model, train_loader, val_loader, epochs=5, lr=1e-3, device='cpu'):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    best_val_acc, best_state_dict = 0.0, None

    for epoch in range(1, epochs+1):
        model.train()
        total_train_loss = 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        val_loss, all_preds, all_targets = 0, [], []
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                outputs = model(X)
                val_loss += criterion(outputs, y).item()
                preds = torch.argmax(outputs, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(y.cpu().numpy())
        avg_val_loss = val_loss / len(val_loader)
        val_acc = accuracy_score(all_targets, all_preds)
        print(f"Эпоха [{epoch}/{epochs}]: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}, Val Acc = {val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state_dict = model.state_dict()

    if best_state_dict:
        model.load_state_dict(best_state_dict)
    return model

def predict_text(model, embeddings, text, embedding_dim, chunk_size=200, step=200, device='cpu'):
    model.eval()
    chunks = chunk_text(text, chunk_size, step)
    if not chunks:
        return 0
    # Вычисляем усреднённый вектор для каждого чанка
    X = np.array([compute_average_vector(chunk, embeddings, embedding_dim) for chunk in chunks])
    X_torch = torch.tensor(X, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_torch)
        # Суммируем логиты по чанкам и выбираем класс с максимальным значением
        return torch.argmax(logits.sum(dim=0)).item()

# Подготовка данных: нарезка текстов с динамическими параметрами

In [47]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

author_files = ["Fry.txt", "Genri.txt", "Simak.txt", "Bulgakov.txt", "Bradbury.txt", "Strugatskie.txt"]
path_to_files = "texts/"
full_paths = [os.path.join(path_to_files, filename) for filename in author_files]
texts, labels, label2author = prepare_author_chunks(full_paths, CHUNK_SIZE, STEP_SIZE)
num_classes = len(label2author)

print("Всего чанков (фрагментов) после нарезки:", len(texts))

# Разбиение на обучающую и валидационную выборки
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=VAL_SIZE, random_state=42, stratify=labels)
print(f"Train chunks: {len(train_texts)}, Val chunks: {len(val_texts)}")

# Вывод примеров чанков для каждого автора
for i in range(len(label2author)):
    sample_idx = labels.index(i)
    print(f"Автор: {label2author[i]}")
    print(texts[sample_idx])
    print("-" * 80)

# Преобразование текстов в векторное представление посредством усреднения векторов предобученных эмбеддингов

Using device: cuda
Всего чанков (фрагментов) после нарезки: 9188
Train chunks: 7350, Val chunks: 1838
Автор: Fry
﻿Власть несбывшегося – С тех пор как меня угораздило побывать в этой грешной Черхавле, мне ежедневно снится какая-то дичь! – сердито сказал я Джуффину. – Сглазили они меня, что ли? А собственно, почему бы и нет!.. – Я даже не стану тратить драгоценное время на то, чтобы тебя успокаивать: ты и сам отлично понимаешь, что метешь чушь! – улыбнулся шеф, заботливо пододвигая ко мне кружку с горячей камрой. – Просто ты не любишь, когда тебя будят на рассвете, и первые полчаса готов ворчать по любому поводу, как старый хрыч, предчувствующий приближение очередного приступа ревматизма… Никто тебя не сглазил, и так называемая дичь снится тебе отнюдь не ежедневно. Ну разве что сегодня, если не врешь… И поделом, между прочим! Нечего так беззастенчиво дрыхнуть на рабочем месте. – Все претензии к внезапно угомонившимся друзьям вашей бурной юности, – проворчал я. – Я же не виноват, что они 

# Формирование датасетов, создание модели

In [ ]:
X_train = np.array([compute_average_vector(text, embeddings_dict, EMBEDDING_DIM) for text in train_texts])
X_val = np.array([compute_average_vector(text, embeddings_dict, EMBEDDING_DIM) for text in val_texts])

train_dataset = TextDataset(X_train, train_labels)
val_dataset = TextDataset(X_val, val_labels)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

input_dim = EMBEDDING_DIM  # размер входного слоя соответствует размерности эмбеддингов

# Создаем модель MLP
model = MLP(input_dim, hidden_dim1=298, hidden_dim2=252, output_dim=num_classes, dropout_rate=0.2).to(device)

# Обучение модели

In [ ]:
print("\n--- Обучение модели ---")
model = train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, device=device)


--- Обучение модели ---
Эпоха [1/70]: Train Loss = 1.4293, Val Loss = 1.0654, Val Acc = 0.5490
Эпоха [2/70]: Train Loss = 0.9515, Val Loss = 0.8072, Val Acc = 0.7100
Эпоха [3/70]: Train Loss = 0.6280, Val Loss = 0.4880, Val Acc = 0.8357
Эпоха [4/70]: Train Loss = 0.4382, Val Loss = 0.3480, Val Acc = 0.8868
Эпоха [5/70]: Train Loss = 0.3631, Val Loss = 0.2820, Val Acc = 0.9032
Эпоха [6/70]: Train Loss = 0.3042, Val Loss = 0.2383, Val Acc = 0.9200
Эпоха [7/70]: Train Loss = 0.2784, Val Loss = 0.2545, Val Acc = 0.9119
Эпоха [8/70]: Train Loss = 0.2433, Val Loss = 0.2201, Val Acc = 0.9211
Эпоха [9/70]: Train Loss = 0.2170, Val Loss = 0.2757, Val Acc = 0.9075
Эпоха [10/70]: Train Loss = 0.2184, Val Loss = 0.2115, Val Acc = 0.9282
Эпоха [11/70]: Train Loss = 0.2002, Val Loss = 0.2010, Val Acc = 0.9325
Эпоха [12/70]: Train Loss = 0.1895, Val Loss = 0.1894, Val Acc = 0.9407
Эпоха [13/70]: Train Loss = 0.1757, Val Loss = 0.1769, Val Acc = 0.9423
Эпоха [14/70]: Train Loss = 0.1710, Val Loss = 0

# Оценка модели на валидационной выборке

In [ ]:
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for X, y in val_loader:
        X, y = X.to(device), y.to(device)
        preds = torch.argmax(model(X), dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(y.cpu().numpy())
val_acc = accuracy_score(all_true, all_preds)
print("\n=== Итоговая оценка на валидационной выборке ===")
print(f"Val Accuracy: {val_acc:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(all_true, all_preds))
print("\nClassification Report:")
print(classification_report(all_true, all_preds, target_names=[label2author[i] for i in range(num_classes)]))


=== Итоговая оценка на валидационной выборке ===
Val Accuracy: 0.9548
Confusion Matrix:
[[569   1   0   0   3  20]
 [  1 162   0   0   0   1]
 [ 11   3 237   0   1  12]
 [  0   0   0 252   0  14]
 [  2   3   0   0 211   9]
 [  1   0   0   0   1 324]]

Classification Report:
              precision    recall  f1-score   support

         Fry       0.97      0.96      0.97       593
       Genri       0.96      0.99      0.97       164
       Simak       1.00      0.90      0.95       264
    Bulgakov       1.00      0.95      0.97       266
    Bradbury       0.98      0.94      0.96       225
 Strugatskie       0.85      0.99      0.92       326

    accuracy                           0.95      1838
   macro avg       0.96      0.95      0.96      1838
weighted avg       0.96      0.95      0.96      1838



# Классификация тестов

In [ ]:
df = pd.read_csv("author_classification.csv")
test_true = []
test_pred = []
test_files_dir = "texts/"

print("\n--- Классификация тестовых файлов (на уже обученной модели) ---")
for _, row in df.iterrows():
    fname = os.path.join(test_files_dir, row['filename'])
    true_author = row['author']
    
    if not os.path.exists(fname):
        print(f"Файл {fname} не найден, пропуск...")
        continue
    
    text = read_file(fname)
    pred_label = predict_text(model, embeddings_dict, text, EMBEDDING_DIM, chunk_size=CHUNK_SIZE, step=STEP_SIZE, device=device)
    pred_author = label2author[pred_label]
    
    test_true.append(true_author)
    test_pred.append(pred_author)
    
    print(f"Файл {fname} -> предсказанный автор: {pred_author} (истинный: {true_author})")

print("\n=== Оценка на тестовом наборе ===")
print("Accuracy:", accuracy_score(test_true, test_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(test_true, test_pred, labels=list(label2author.values())))
print("\nClassification Report:")
print(classification_report(test_true, test_pred, labels=list(label2author.values())))


--- Классификация тестовых файлов (на уже обученной модели) ---
Файл texts/author1.txt -> предсказанный автор: Genri (истинный: Genri)
Файл texts/author2.txt -> предсказанный автор: Simak (истинный: Simak)
Файл texts/author3.txt -> предсказанный автор: Genri (истинный: Genri)
Файл texts/author4.txt -> предсказанный автор: Bulgakov (истинный: Bulgakov)
Файл texts/author5.txt -> предсказанный автор: Fry (истинный: Genri)
Файл texts/author6.txt -> предсказанный автор: Bradbury (истинный: Bradbury)
Файл texts/author7.txt -> предсказанный автор: Fry (истинный: Fry)
Файл texts/author8.txt -> предсказанный автор: Fry (истинный: Fry)
Файл texts/author9.txt -> предсказанный автор: Strugatskie (истинный: Strugatskie)
Файл texts/author10.txt -> предсказанный автор: Bradbury (истинный: Bradbury)
Файл texts/author11.txt -> предсказанный автор: Bulgakov (истинный: Bulgakov)
Файл texts/author12.txt -> предсказанный автор: Strugatskie (истинный: Bradbury)
Файл texts/author13.txt -> предсказанный авто

# Анализ покрытия эмбеддингов (train)

In [ ]:
print("\n=== Проверка покрытия эмбеддингов на обучающем тексте ===")
check_embedding_coverage(train_texts, embeddings_dict)


=== Проверка покрытия эмбеддингов на обучающем тексте ===


KeyboardInterrupt: 